# 🏆 Ultimate Challenge: Ultimate Challenge Dataset v3

This is the **capstone notebook** that showcases the entity resolution system's ultimate capabilities on the most challenging dataset available - the Ultimate Challenge dataset. This notebook demonstrates multilingual entity matching across 6+ scripts, complex cultural naming conventions, and sophisticated business entity resolution.

## 📊 About the Ultimate Challenge Dataset

**What is the Ultimate Challenge Dataset?**

The Ultimate Challenge dataset is the most challenging dataset available for evaluating entity resolution systems. It contains entities, articles, and unique challenge types covering multilingual, cultural, and business scenarios. The exact counts are shown dynamically when you load the dataset (see Section 2 below).

**About the Comprehensive Evaluation Tiered System**

This dataset is part of a tiered evaluation system (Tier 0-5) that progressively increases in complexity:
- **Tier 0**: Simple exact matches
- **Tier 1**: Basic variations (nicknames, abbreviations)
- **Tier 2**: Moderate complexity (aliases, titles)
- **Tier 3**: Advanced matching (cross-script, transliteration)
- **Tier 4**: Complex scenarios (cultural variations, compound names)
- **Tier 5**: Ultimate Challenge (this dataset) - Most challenging scenarios

**Why This Naming?**

While this dataset is labeled "Tier 5" in the comprehensive evaluation system, we refer to it as the "Ultimate Challenge dataset" to avoid confusion. In the context of the comprehensive evaluation, it's the most challenging tier (Tier 5), but users don't need to know about the tiered system to use this notebook.

**Explore Other Tiers**

If you want to see progressively complex scenarios or simpler cases, you can explore the comprehensive evaluation directory:
- **Location**: `use_cases/comprehensive_evaluation/`
- **Contents**: All tiered datasets (Tier 0-5) with evaluation scripts
- **Use Case**: Compare system performance across different complexity levels

**Controlled Evaluation Design - Important Limitations**

This evaluation **controls for** entity preparation and article processing, which gives the system significant advantages:

- **What This Means**: We use mock extracted entities from `expected_matches` instead of running the full pipeline
- **Why This Matters**: This isolates the matching/judgment stage - we're testing whether the system can **MATCH** entities correctly, not whether it can **PREPARE** or **EXTRACT** them correctly
- **What's Being Tested**: Entity matching (Elasticsearch search) and LLM judgment (match decision-making)
- **What's NOT Being Tested**: Entity enrichment (Notebook 1) and entity extraction (Notebook 2)

**⚠️ Important: This Design Gives the System Unfair Advantages**

The controlled evaluation design means:
1. **Perfect Extraction**: Extracted entities are created directly from `expected_matches`, so:
   - Only entities that should match are extracted (no false positives from extraction)
   - No entities that shouldn't be in the watch list are extracted
   - Extraction is 100% accurate (not realistic)

2. **Perfect Watch List**: The watch list contains exactly the entities from `expected_matches`, so:
   - Every extracted entity has a corresponding watch list entity
   - No distractor entities in the watch list that might cause false matches
   - The watch list is perfectly aligned with the articles

3. **No Real-World Noise**: We're not testing:
   - Whether extraction would find the right entities in real text
   - Whether extraction would create false positives
   - Whether the watch list contains distractor entities

**Why This Design Exists**: This isolates matching/judgment quality from extraction/preparation quality. It's like testing a car's handling on a perfect test track - we control the conditions to focus on one aspect of performance.

**What This Means for Results**: The high precision/recall (90%+) is **expected** given this controlled design. In a real-world scenario with imperfect extraction and a larger watch list with distractors, performance would likely be lower.

## 🎯 Goals

By the end of this notebook, you will:

- Understand the ultimate challenge dataset (see Section 2 for exact counts)
- See the system handle multilingual matching (Japanese, Arabic, Hebrew, Chinese, Cyrillic, Korean, Latin)
- Evaluate quality metrics (precision, recall, F1) on extreme complexity scenarios
- Explore showcase examples from selected challenge types
- Analyze performance across scripts, challenge categories, and match complexity

## 📚 What You'll Learn

1. **Direct Matching Approach**: How to bypass `RealTimeEntityMatcher` for better event loop handling
2. **Mock Entity Extraction**: Controlled evaluation using `expected_matches` from test data
3. **Quality Evaluation**: Precision, recall, and F1 metrics calculation
4. **Challenge Type Showcase**: Visual examples of multilingual and cross-script matching
5. **Performance Analysis**: Breakdowns by script, challenge category, and complexity

## ✅ Prerequisites

- **Required**: Elasticsearch connection with E5 model for semantic search
- **Required**: OpenAI API key for LLM-powered match judgment
- **Note**: This notebook uses a controlled evaluation approach - entity preparation and article processing are handled via mock extraction from `expected_matches` to focus purely on entity matching and judging capabilities.

## 📋 What Makes This Dataset Challenging?

**💡 Note for Non-Multilingual Readers:**

This notebook includes examples with non-Latin characters (Chinese, Japanese, Arabic, Hebrew, Cyrillic, Korean). **You don't need to read these characters to understand the concepts!** We'll always provide:
- **Transliterations** (Latin script versions) alongside non-Latin characters
- **English explanations** of what the characters represent
- **Focus on the concept** (e.g., "name order variations") rather than the specific characters

For example, when you see "安倍晋三 (Shinzo Abe)", the important part is understanding that this demonstrates "multiple writing systems" - you don't need to read the Japanese characters.

---

This dataset is challenging because it tests the system's ability to handle:

**Multilingual Challenges:**
- **Cross-Script Matching**: Matching entities across different writing systems (e.g., Cyrillic "Лев Толстой" (Leo Tolstoy) vs Latin "Leo Tolstoy")
- **Transliteration Variations**: Multiple valid transliterations of the same name (e.g., "北京" (Beijing) vs "Beijing" vs "Peking")
- **Language-Specific Patterns**: Different name order conventions (Japanese surname-first vs English given-first)

**Cultural Variations:**
- **Name Order Differences**: Some cultures write surnames first, others write given names first
- **Patronymics**: Middle names that indicate family relationships (e.g., Russian patronymics)
- **Honorifics**: Titles and honorifics that vary by culture and context

**Complex Entity Relationships:**
- **Title + Name Matching**: Matching titles like "The President" to specific people based on context
- **Compound Entities**: Entities that combine multiple concepts (e.g., "Tesla CEO" → "Elon Musk")
- **Ambiguous References**: References that could refer to multiple entities without context

**Why These Challenges Matter**

In real-world entity resolution:
- **Multilingual Data**: News articles, social media, and documents often contain multiple languages
- **Cultural Diversity**: Global systems need to handle names from many different cultures
- **Context Dependency**: The same reference can mean different things in different contexts

**What Makes Entity Resolution Hard in These Cases**

- **No Exact Matches**: The extracted entity and watched entity rarely match exactly
- **Context Required**: The system needs to understand context to make correct matches
- **Ambiguity**: Multiple entities might match, requiring sophisticated judgment
- **Cross-Language**: The system must understand meaning across languages and scripts

---

## 📋 Dataset Overview

**Ultimate Challenge Dataset** characteristics:
- **Entities** across multiple scripts and challenge types
- **Articles** with varying complexity levels
- **Unique challenge types** (from articles) covering multilingual, cultural, and business scenarios
- **Multiple scripts**: Japanese, Arabic, Hebrew, Chinese, Cyrillic, Korean, Latin

*Note: Exact counts are calculated dynamically when you load the dataset (see Section 2 below).*


## 1. Setup & Prerequisites


In [ ]:
import sys
import os
import json
import time
import asyncio
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Configure logging to show only warnings and errors
logging.basicConfig(level=logging.WARNING, force=True)

# Suppress verbose logging from various libraries
loggers_to_suppress = [
    "entity_resolution_demo",
    "entity_resolution_demo.entity_matching",
    "entity_resolution_demo.entity_matching.minimal_function_calling_judge",
    "entity_resolution_demo.entity_matching.enhanced_batch_match_judge",
    "entity_resolution_demo.entity_preparation",
    "entity_resolution_demo.entity_preparation.entity_enricher",
    "entity_resolution_demo.entity_preparation.entity_indexer",
    "entity_resolution_demo.entity_preparation.entity_watch_list",
    "entity_resolution_demo.search",
    "entity_resolution_demo.search.elastic_client",
    "entity_resolution_demo.pipeline_runner",
    "entity_resolution_demo.pipeline_runner.utils",
    "elastic_transport",
    "elastic_transport.transport",
    "elasticsearch",
    "urllib3",
    "urllib3.connectionpool",
    "requests",
    "requests.packages.urllib3",
    "httpx",
    "httpcore"
]

for logger_name in loggers_to_suppress:
    logging.getLogger(logger_name).setLevel(logging.WARNING)
    logging.getLogger(logger_name).propagate = False

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables first (if using .env file)
from dotenv import load_dotenv
load_dotenv()

# Import project modules
from entity_resolution_demo.pipeline_runner.config import load_config
from entity_resolution_demo.search.elastic_client import ElasticClient
from entity_resolution_demo.entity_preparation.entity_watch_list import EntityWatchList, WatchedEntity
from entity_resolution_demo.entity_preparation.entity_enricher import EntityEnricher
from entity_resolution_demo.entity_preparation.entity_indexer import EntityIndexer
from entity_resolution_demo.entity_matching.elasticsearch_entity_matcher import ElasticsearchEntityMatcher
from entity_resolution_demo.entity_matching.minimal_function_calling_judge import MinimalFunctionCallingJudge
from entity_resolution_demo.article_processing.article_processor import Article, ProcessedArticle, ExtractedEntity

print("✅ Imports successful")
print("ℹ️  Logging configured to show only warnings and errors")


In [ ]:
# Load configuration
config = load_config()

# Verify dependencies
print("🔍 Verifying dependencies...")

# Check Elasticsearch connection
try:
    elastic_client = ElasticClient(config)
    elastic_client.es.ping()
    print("✅ Elasticsearch connection verified")
except Exception as e:
    print(f"❌ Elasticsearch connection failed: {e}")
    raise

# Check OpenAI API key (basic check)
openai_key = os.getenv('OPENAI_API_KEY')
if openai_key:
    print("✅ OpenAI API key found")
else:
    print("⚠️ OpenAI API key not found in environment variables")
    print("   Make sure to set OPENAI_API_KEY in your .env file or environment")

print("\n✅ All prerequisites verified")


## 2. Load Ultimate Challenge Data


In [ ]:
# Load Tier 5 test articles and watch list
from pathlib import Path
import json

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "comprehensive_evaluation" / "data"

with open(data_dir / 'tier5_test_articles_v2.json', 'r', encoding='utf-8') as f:
    tier5_articles_data = json.load(f)

with open(data_dir / 'tier5_watch_list_cleaned.json', 'r', encoding='utf-8') as f:
    tier5_watch_list_data = json.load(f)

# Handle nested structure
if isinstance(tier5_articles_data, dict) and 'articles' in tier5_articles_data:
    tier5_articles = tier5_articles_data['articles']
else:
    tier5_articles = tier5_articles_data

# Load watch list (using cleaned version with only legitimate aliases)
with open(data_dir / 'tier5_watch_list_cleaned.json', 'r', encoding='utf-8') as f:
    tier5_watch_list_data = json.load(f)

# Handle nested structure
if isinstance(tier5_watch_list_data, dict) and 'entities' in tier5_watch_list_data:
    tier5_entities = tier5_watch_list_data['entities']
else:
    tier5_entities = tier5_watch_list_data

print(f"✅ Loaded Tier 5 Ultimate Challenge data:")
print(f"   - Articles: {len(tier5_articles)}")
print(f"   - Entities: {len(tier5_entities)}")

# Calculate statistics
scripts = set()
challenge_types = set()
for entity in tier5_entities:
    scripts.add(entity.get('script', 'unknown'))
    challenge_types.add(entity.get('challenge_type', 'unknown'))

# Count challenge types from articles (test_category) and entities (challenge_type)
article_challenge_types = {a.get('test_category', '') for a in tier5_articles}
entity_challenge_types = {e.get('challenge_type', '') for e in tier5_entities}

print(f"   - Scripts: {len(scripts)} ({', '.join(sorted(scripts))})")
print(f"   - Challenge types (from articles): {len(article_challenge_types)}")
print(f"   - Challenge types (from entities): {len(entity_challenge_types)}")

# Helper function to get transliteration from entity aliases
def get_transliteration(entity_name, entities_list):
    """Get Latin transliteration from entity aliases"""
    for entity in entities_list:
        if entity['name'] == entity_name:
            aliases = entity.get('aliases', [])
            # Find first alias that's mostly Latin (ASCII) characters
            for alias in aliases:
                if alias and all(ord(c) < 128 for c in alias):
                    return alias
    return None

# Show sample entities with transliterations
print("\n📋 Sample Entities:")
print("💡 Note: Non-Latin characters are shown with their transliterations (Latin script versions)")
print("   You don't need to read the original characters to understand the concepts!\n")
for entity in tier5_entities[:3]:
    entity_name = entity['name']
    script = entity.get('script', 'unknown')
    challenge_type = entity.get('challenge_type', 'unknown')
    transliteration = get_transliteration(entity_name, tier5_entities)
    
    if transliteration and entity_name != transliteration:
        print(f"   • {entity_name} ({script}) = '{transliteration}' - {challenge_type}")
    else:
        print(f"   • {entity_name} ({script}) - {challenge_type}")


In [ ]:
# Helper function to get transliteration (reuse from previous cell)
def get_transliteration(entity_name, entities_list):
    """Get Latin transliteration from entity aliases"""
    for entity in entities_list:
        if entity['name'] == entity_name:
            aliases = entity.get('aliases', [])
            for alias in aliases:
                if alias and all(ord(c) < 128 for c in alias):
                    return alias
    return None

# Show sample articles with transliterations
print("📰 Sample Articles:")
for article in tier5_articles[:2]:
    print(f"\n   Article ID: {article['id']}")
    print(f"   Title: {article.get('title', 'N/A')}")
    print(f"   Language: {article.get('language', 'N/A')}")
    print(f"   Complexity: {article.get('complexity', 'N/A')}")
    print(f"   Expected matches: {len(article.get('expected_matches', []))}")
    if article.get('expected_matches'):
        extracted = article['expected_matches'][0].get('extracted_entity', '')
        watched = article['expected_matches'][0].get('watch_list_entity', '')
        extracted_translit = get_transliteration(extracted, tier5_entities)
        watched_translit = get_transliteration(watched, tier5_entities)
        
        # Show match with transliterations
        if extracted_translit and extracted != extracted_translit:
            extracted_display = f"'{extracted}' ({extracted_translit})"
        else:
            extracted_display = f"'{extracted}'"
        
        if watched_translit and watched != watched_translit:
            watched_display = f"'{watched}' ({watched_translit})"
        else:
            watched_display = f"'{watched}'"
        
        print(f"   Sample match: {extracted_display} → {watched_display}")


## 3. Prepare Data for Matching

This section converts the Ultimate Challenge dataset into the formats required for entity matching:
- **Watch List**: Convert Ultimate Challenge entities to `EntityWatchList` format with `explicit_context`
- **Articles**: Convert Ultimate Challenge articles to `ProcessedArticle` format using mock extraction from `expected_matches`

**⚠️ Controlled Evaluation Approach - Creates Unfair Advantages**

We use `expected_matches` from the test articles to create mock extracted entities. This means:

**What We're Doing:**
- Creating extracted entities directly from `expected_matches` (the "correct answers")
- Only extracting entities that we already know should match
- Using a watch list that contains exactly the entities from `expected_matches`

**Why This Is Unfair:**
1. **Perfect Extraction**: We're only extracting entities that should match - no false positives from extraction
2. **Perfect Watch List**: The watch list contains exactly the entities we're looking for - no distractors
3. **No Real-World Challenges**: We're not testing whether extraction would find the right entities, or whether the watch list contains entities that might cause false matches

**What This Means:**
- The high precision/recall (90%+) is **expected** given this controlled design
- In a real-world scenario with imperfect extraction and a larger watch list, performance would likely be lower
- This evaluation tests matching/judgment in isolation, not the full pipeline

**Why We Do This:**
This isolates matching/judgment quality from extraction/preparation quality, allowing us to focus on one aspect of the system's performance.


In [ ]:
# Convert Ultimate Challenge watch list to EntityWatchList format
print("📋 Creating EntityWatchList from Ultimate Challenge dataset...")

watch_list = EntityWatchList()
duplicate_count = 0

for entity_data in tier5_entities:
    entity_name = entity_data['name']
    
    # Check if entity already exists - skip duplicates gracefully
    existing_entity = watch_list.get_entity_by_name(entity_name)
    if existing_entity:
        # Entity already exists - skip this duplicate
        duplicate_count += 1
        continue
    
    # Extract explicit context if available
    explicit_context = entity_data.get('explicit_context', '')
    
    # Prepare metadata with context
    metadata = entity_data.get('metadata', {})
    if explicit_context:
        metadata['explicit_context'] = explicit_context
    
    # Get aliases if available
    aliases = entity_data.get('aliases', [])
    
    # Add entity to watch list with aliases
    entity_id = watch_list.add_entity(
        name=entity_name,
        entity_type=entity_data.get('entity_type', 'PERSON').upper(),
        aliases=aliases,
        metadata=metadata
    )
    
    # Check if add_entity returned False (duplicate - shouldn't happen after check above)
    if entity_id is False:
        duplicate_count += 1

print(f"✅ Created watch list with {len(watch_list.get_all_entities())} entities")
if duplicate_count > 0:
    print(f"   Skipped {duplicate_count} duplicate entities")
print(f"   Total entities with aliases: {sum(1 + len(entity_data.get('aliases', [])) for entity_data in tier5_entities)}")


In [ ]:
# Enrich and index entities
print("🔍 Enriching and indexing entities...")

entity_enricher = EntityEnricher(config)
entity_indexer = EntityIndexer(elastic_client, config)

# Get the actual index name used by EntityIndexer
# This is important so ElasticsearchEntityMatcher can use the same index
actual_index_name = entity_indexer.entity_index
watch_list.index_name = actual_index_name
print(f"   Using index name: {actual_index_name}")

enriched_entities = []
for entity in watch_list.get_all_entities():
    # Use explicit context if available in metadata
    metadata = entity.metadata or {}
    explicit_context = metadata.get('explicit_context', '')
    
    # Enrich entity - pass name string, not the entity object
    # enrich_entity expects: (name: str, source_context: str = None, aliases: List[str] = None)
    enriched = entity_enricher.enrich_entity(
        name=entity.name,
        source_context=explicit_context if explicit_context else None,
        aliases=entity.aliases if entity.aliases else None
    )
    
    # Override context with explicit context if available
    if explicit_context:
        enriched.entity_context = explicit_context
    
    enriched_entities.append(enriched)

# Create indices first (important!)
entity_indexer.create_indices()

# Index entities
indexing_results = []
for enriched in enriched_entities:
    result = entity_indexer.index_entity(enriched)
    indexing_results.append(result)

print(f"✅ Enriched and indexed {len(enriched_entities)} entities")
print(f"   Successful indexings: {sum(1 for r in indexing_results if r)}")
print(f"   Index name: {actual_index_name}")


In [ ]:
# Convert Tier 5 test articles to ProcessedArticle format (mock extraction from expected_matches)
print("📰 Converting Tier 5 articles to ProcessedArticle format...")

processed_articles = []

for article_data in tier5_articles:
    try:
        # Create Article object
        article = Article(
            id=article_data['id'],
            title=article_data.get('title', ''),
            content=article_data.get('content', ''),
            source=article_data.get('source', 'Tier5 Challenge Data'),
            language=article_data.get('language', 'en'),
            url=f"test://example.com/{article_data['id']}"
        )
        
        # Mock entity extraction using expected_matches (controlled evaluation)
        extracted_entities = []
        expected_matches = article_data.get('expected_matches', [])
        
        for i, match in enumerate(expected_matches):
            # Determine entity type from watch list entity (if available)
            watch_list_entity_name = match.get('watch_list_entity', '')
            entity_type = "PERSON"  # Default
            
            # If watch_list_entity is empty, it means no match is expected
            # Still create the extracted entity so it can be tested
            if watch_list_entity_name:
                for entity in tier5_entities:
                    if entity['name'] == watch_list_entity_name:
                        entity_type = entity.get('entity_type', 'PERSON').upper()
                        break
            
            # Create extracted entity (even if no match is expected)
            extracted_entity = ExtractedEntity(
                name=match['extracted_entity'],
                entity_type=entity_type,
                confidence=1.0,  # Perfect extraction (controlled)
                context=f"From article: {article_data.get('title', '')[:100]}...",
                position=i,
                extraction_method="controlled_evaluation"
            )
            extracted_entities.append(extracted_entity)
        
        # Create ProcessedArticle
        processed_article = ProcessedArticle(
            article=article,
            extracted_entities=extracted_entities,
            processing_time=0.1,  # Mock processing time
            total_entities_found=len(extracted_entities),
            unique_entities=set(e.name for e in extracted_entities)
        )
        
        processed_articles.append(processed_article)
        
    except Exception as e:
        print(f"⚠️ Warning: Failed to process article {article_data.get('id', 'unknown')}: {e}")
        continue

print(f"✅ Processed {len(processed_articles)} articles")
print(f"   Total extracted entities: {sum(len(p.extracted_entities) for p in processed_articles)}")


## 4. Initialize Direct Matching Components

**Why Direct Approach?** We bypass `RealTimeEntityMatcher` because it has event loop conflicts with `MinimalFunctionCallingJudge`'s async methods. When `MinimalFunctionCallingJudge` uses async methods, calling `asyncio.run()` from a running event loop (common in Jupyter environments) causes kernel restarts. The direct approach uses a thread-based execution pattern: we call `ElasticsearchEntityMatcher` and `MinimalFunctionCallingJudge` directly, running async code in separate threads with their own event loops to avoid conflicts.


In [ ]:
# Initialize ElasticsearchEntityMatcher
print("🔍 Initializing ElasticsearchEntityMatcher...")

es_matcher = ElasticsearchEntityMatcher(
    watch_list=watch_list,
    elastic_client=elastic_client,
    config=config
)

print("✅ ElasticsearchEntityMatcher initialized")


In [ ]:
# Initialize MinimalFunctionCallingJudge
print("⚖️ Initializing MinimalFunctionCallingJudge...")

minimal_judge = MinimalFunctionCallingJudge(config=config)

print("✅ MinimalFunctionCallingJudge initialized")


In [ ]:
# Verify components are ready
print("✅ Components ready for matching:")
print(f"   - ElasticsearchEntityMatcher: Ready")
print(f"   - MinimalFunctionCallingJudge: Ready")
print(f"   - Watch list: {len(watch_list.get_all_entities())} entities")
print(f"   - Processed articles: {len(processed_articles)} articles")


## 5. Run Ultimate Challenge Matching (Direct Approach)

This section processes all articles using the direct approach:
1. Use `ElasticsearchEntityMatcher.find_potential_matches()` to find candidates
2. Collect all `PotentialMatch` objects for each article
3. Convert `PotentialMatch` to batch format: `{'query_name', 'candidate_name', 'context'}`
4. Use async context to call `MinimalFunctionCallingJudge.judge_batch()`
5. Convert `MinimalNameMatchResult` back to match format with `article_id`


In [ ]:
# Run matching using direct approach
print("🚀 Running Ultimate Challenge matching (direct approach)...")
print("=" * 60)
print("ℹ️  Using thread-based approach to avoid kernel restarts")
print("   (Async code runs in separate threads with their own event loops)")

start_time = time.time()
all_results = []

# Process each article
for i, processed_article in enumerate(processed_articles):
    article_id = processed_article.article.id
    print(f"\nProcessing article {i+1}/{len(processed_articles)}: {article_id}")
    
    try:
        # Step 1: Find potential matches using Elasticsearch
        # find_potential_matches requires (extracted_entity, article) as arguments
        # So we need to iterate through extracted entities
        potential_matches = []
        for extracted_entity in processed_article.extracted_entities:
            matches = es_matcher.find_potential_matches(extracted_entity, processed_article.article)
            potential_matches.extend(matches)
        
        if not potential_matches:
            print(f"   ⚠️ No potential matches found")
            continue
        
        print(f"   Found {len(potential_matches)} potential matches")
        
        # Step 2 & 3: Convert PotentialMatch to batch format
        batch_input = []
        for potential_match in potential_matches:
            batch_input.append({
                'query_name': potential_match.extracted_entity.name,
                'candidate_name': potential_match.watched_entity.name,
                'context': potential_match.extracted_entity.context or ""
            })
        
        # Step 4: Judge batch using thread-based approach to avoid kernel restarts
        # Use ThreadPoolExecutor to run async code in a separate thread with its own event loop
        # This prevents kernel restarts that occur when calling asyncio.run() from a running event loop
        import concurrent.futures
        
        def run_judge_batch_in_thread():
            """Run async judge_batch in a separate thread with its own event loop"""
            def run_in_thread():
                # Create a new event loop in this thread
                loop = asyncio.new_event_loop()
                asyncio.set_event_loop(loop)
                try:
                    return loop.run_until_complete(minimal_judge.judge_batch(batch_input))
                finally:
                    loop.close()
            
            # Run in a separate thread to avoid event loop conflicts
            with concurrent.futures.ThreadPoolExecutor() as executor:
                future = executor.submit(run_in_thread)
                return future.result()
        
        batch_results = run_judge_batch_in_thread()
        
        # Helper function to get transliteration (reuse from earlier cells)
        def get_transliteration(entity_name, entities_list):
            """Get Latin transliteration from entity aliases"""
            for entity in entities_list:
                if entity['name'] == entity_name:
                    aliases = entity.get('aliases', [])
                    for alias in aliases:
                        if alias and all(ord(c) < 128 for c in alias):
                            return alias
            return None
        
        # Step 5: Convert results back to match format and show detailed output
        article_matches = []
        for k, result in enumerate(batch_results or []):
            # Normalize result to dict
            if hasattr(result, 'model_dump'):
                res = result.model_dump()
            elif isinstance(result, dict):
                res = result
            else:
                res = {}
            
            # Create match result
            extracted = batch_input[k]['query_name']
            watched = batch_input[k]['candidate_name']
            match_result = {
                'article_id': article_id,
                'extracted_entity': extracted,
                'watched_entity': watched,
                'confidence': res.get('confidence', 0.0),
                'is_match': res.get('is_match', False),
                'match_type': res.get('match_type', 'unknown'),
                'reasoning': res.get('reasoning', '')
            }
            
            all_results.append(match_result)
            article_matches.append(match_result)
        
        # Show detailed match information
        confirmed = sum(1 for m in article_matches if m.get('is_match', False))
        print(f"   ✅ Processed {len(article_matches)} matches, {confirmed} confirmed")
        
        # Show detailed output for selected interesting articles that demonstrate different challenge types
        # These articles showcase: multilingual matching, cross-script transliteration, and business entity resolution
        interesting_article_ids = [
            'tier5_japanese_name_order_001',  # Japanese multilingual example
            'tier5_cross_script_tolstoy_001',   # Cross-script (Cyrillic/Latin) example
            'tier5_business_hierarchy_001'     # Business entity resolution example
        ]
        
        if article_id in interesting_article_ids:
            print(f"\n   📋 Match Details (Example Article):")
            for match in article_matches:
                extracted = match['extracted_entity']
                watched = match['watched_entity']
                is_match = match.get('is_match', False)
                confidence = match.get('confidence', 0.0)
                match_type = match.get('match_type', 'unknown')
                reasoning = match.get('reasoning', '')
                
                # Get transliterations for non-Latin characters
                extracted_translit = get_transliteration(extracted, tier5_entities)
                watched_translit = get_transliteration(watched, tier5_entities)
                
                # Format entity names with transliterations
                if extracted_translit and extracted != extracted_translit:
                    extracted_display = f"'{extracted}' ({extracted_translit})"
                else:
                    extracted_display = f"'{extracted}'"
                
                if watched_translit and watched != watched_translit:
                    watched_display = f"'{watched}' ({watched_translit})"
                else:
                    watched_display = f"'{watched}'"
                
                # Show match status
                status = "✅ CONFIRMED" if is_match else "❌ REJECTED"
                print(f"      {status}: {extracted_display} → {watched_display}")
                print(f"         Confidence: {confidence:.2f} | Type: {match_type}")
                if reasoning:
                    reasoning_preview = reasoning[:200] + "..." if len(reasoning) > 200 else reasoning
                    print(f"         Reasoning: {reasoning_preview}")
                print()
        
    except Exception as e:
        print(f"   ❌ Error processing article {article_id}: {e}")
        import traceback
        traceback.print_exc()
        continue

execution_time = time.time() - start_time

print("\n" + "=" * 60)
print(f"✅ Matching completed in {execution_time:.2f} seconds")
print(f"   Total matches processed: {len(all_results)}")
print(f"   Confirmed matches: {sum(1 for r in all_results if r.get('is_match', False))}")
print(f"   Articles processed: {len(processed_articles)}")

# Save results to state file for inspection
import json
from pathlib import Path
from datetime import datetime

results_state = {
    'execution_time': execution_time,
    'total_matches': len(all_results),
    'confirmed_matches': sum(1 for r in all_results if r.get('is_match', False)),
    'articles_processed': len(processed_articles),
    'results': all_results,
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'metadata': {
        'pipeline': 'ultimate_challenge_v3',
        'dataset': 'tier5_ultimate_challenge',
        'source': '05_ultimate_challenge_v3.ipynb'
    }
}

# Save to pipeline_state directory (consistent with other notebooks)
state_dir = Path('pipeline_state')
state_dir.mkdir(exist_ok=True)
state_file = state_dir / 'ultimate_challenge_results_state.json'

with open(state_file, 'w', encoding='utf-8') as f:
    json.dump(results_state, f, indent=2, ensure_ascii=False)

print(f"\n💾 Results saved to: {state_file}")
print(f"   You can inspect the results by loading this JSON file")


## 6. Quality Evaluation

This section evaluates the quality of our entity matching using precision, recall, and F1 metrics. We use `expected_matches` from the test articles as the golden standard (ground truth).

**⚠️ Important Context for Interpreting Results:**

The high precision/recall (90%+) shown here is **expected** given the controlled evaluation design:
- **Perfect Extraction**: We're only testing entities that we know should match (from `expected_matches`)
- **Perfect Watch List**: The watch list contains exactly the entities we're looking for (no distractors)
- **No Real-World Noise**: We're not testing extraction quality or watch list completeness

**What This Means:**
- These results test matching/judgment quality in isolation
- In a real-world scenario with imperfect extraction and a larger watch list, performance would likely be lower
- The results demonstrate the system's matching/judgment capabilities, not the full pipeline's performance

**For Real-World Performance:**
To evaluate the full pipeline, you would need to:
1. Run actual entity extraction (Notebook 2) on the articles
2. Use a larger watch list with distractor entities
3. Test whether extraction finds the right entities and whether the watch list causes false matches


In [ ]:
# Load expected_matches from test articles as golden standard
print("📊 Loading golden standard (expected_matches) from test articles...")

expected_matches_dict = {}
expected_no_matches_dict = {}  # Track cases where no match is expected

for article_data in tier5_articles:
    article_id = article_data['id']
    expected_matches = article_data.get('expected_matches', [])
    
    # Create lookup dictionary: extracted_entity -> watch_list_entity
    article_expected = {}
    article_no_match = []  # Track extracted entities where no match is expected
    
    for match in expected_matches:
        extracted = match.get('extracted_entity', '')
        watched = match.get('watch_list_entity', '')
        
        if extracted and watched:
            # Match is expected
            article_expected[extracted] = watched
        elif extracted and not watched:
            # No match is expected (empty watch_list_entity)
            article_no_match.append(extracted)
    
    if article_expected:
        expected_matches_dict[article_id] = article_expected
    
    if article_no_match:
        expected_no_matches_dict[article_id] = article_no_match

total_expected = sum(len(matches) for matches in expected_matches_dict.values())
total_no_match_expected = sum(len(entities) for entities in expected_no_matches_dict.values())
print(f"✅ Loaded golden standard: {total_expected} expected matches across {len(expected_matches_dict)} articles")
if total_no_match_expected > 0:
    print(f"   Also tracking {total_no_match_expected} extracted entities where no match is expected (e.g., office vs. person)")


In [ ]:
# Calculate quality metrics using comprehensive evaluation approach
print("\n📈 Calculating quality metrics...")
print("=" * 60)

# Create expected_matches lookup with keys: "extracted_entity -> watch_list_entity"
expected_matches_set = set()
for article_id, matches in expected_matches_dict.items():
    for extracted, watched in matches.items():
        key = f"{extracted} -> {watched}"
        expected_matches_set.add(key)

# Create set of extracted entities where no match is expected
expected_no_match_set = set()
for article_id, entities in expected_no_matches_dict.items():
    for extracted in entities:
        expected_no_match_set.add(extracted)

# Create predicted matches set (only confirmed matches)
predicted_matches_set = set()
predicted_matched_entities = set()  # Track which extracted entities got matched
for result in all_results:
    if result.get('is_match', False):
        extracted = result.get('extracted_entity', '')
        watched = result.get('watched_entity', '')
        if extracted and watched:
            key = f"{extracted} -> {watched}"
            predicted_matches_set.add(key)
            predicted_matched_entities.add(extracted)

# Calculate metrics for expected matches
tp = len(expected_matches_set & predicted_matches_set)  # True positives: in both sets
fp = len(predicted_matches_set - expected_matches_set)  # False positives: predicted but not expected
fn = len(expected_matches_set - predicted_matches_set)  # False negatives: expected but not predicted

# Calculate metrics for expected no-matches (cases where no match is expected)
tn = len(expected_no_match_set - predicted_matched_entities)  # True negatives: no match expected and none predicted
fp_no_match = len(expected_no_match_set & predicted_matched_entities)  # False positives: no match expected but one was predicted

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("**Quality Metrics:**")
print(f"   True Positives (TP):  {tp}")
print(f"   False Positives (FP): {fp}")
print(f"   False Negatives (FN): {fn}")
if total_no_match_expected > 0:
    print(f"   True Negatives (TN):  {tn} (no match expected, correctly rejected)")
    print(f"   FP (no-match cases): {fp_no_match} (no match expected, but match was predicted)")
print(f"\n   Precision:            {precision*100:.1f}%")
print(f"   Recall:               {recall*100:.1f}%")
print(f"   F1 Score:             {f1:.3f}")
print(f"\n   Predicted matches:    {len(predicted_matches_set)}")
print(f"   Golden standard:      {len(expected_matches_set)}")
if total_no_match_expected > 0:
    print(f"   No-match cases:       {total_no_match_expected} (correctly rejected: {tn}, incorrectly matched: {fp_no_match})")

print("\n**Quality Insights:**")
print(f"   • Precision: Of matches we predicted, {precision*100:.1f}% were correct")
print(f"   • Recall: Of correct matches in golden standard, we found {recall*100:.1f}%")
print(f"   • F1 Score: {f1:.3f} - balanced measure of precision and recall")

# Compare to expected performance
expected_f1 = 0.85  # >85% F1 for Tier 5
if f1 >= expected_f1:
    print(f"\n   ✅ Performance exceeds expected threshold ({expected_f1:.1%})")
else:
    print(f"\n   ⚠️ Performance below expected threshold ({expected_f1:.1%})")


## 7. Challenge Type Showcase

This section highlights the most impressive and educational matches from 8 selected challenge types (from 50 total in the Ultimate Challenge dataset). We focus on visually impressive multilingual examples, culturally significant naming systems, real-world title matching, and business entity resolution.


In [ ]:
# Select showcase examples for 8 challenge types
print("🎯 Challenge Type Showcase")
print("=" * 60)

# Define selected challenge types
selected_challenge_types = {
    # Multilingual & Cross-Script (3 types)
    'japanese_transliteration': {
        'name': 'Japanese Transliteration',
        'category': 'Multilingual & Cross-Script',
        'description': 'Shows kanji, hiragana, katakana, and romaji variations',
        'example_entity': '安倍晋三'
    },
    'chinese_transliteration': {
        'name': 'Chinese Transliteration',
        'category': 'Multilingual & Cross-Script',
        'description': 'Shows Chinese characters to Latin transliteration',
        'example_entity': '习近平'
    },
    'cross_script_transliteration': {
        'name': 'Cross-Script Transliteration',
        'category': 'Multilingual & Cross-Script',
        'description': 'Shows Cyrillic to Latin script conversion',
        'example_entity': 'Leo Tolstoy'
    },
    # Cultural Naming Systems (2 types)
    'name_order_variations': {
        'name': 'Name Order Variations',
        'category': 'Cultural Naming Systems',
        'description': 'Shows Japanese surname-first vs given-first order',
        'example_entity': '田中太郎'
    },
    'patronymic_systems': {
        'name': 'Patronymic Systems',
        'category': 'Cultural Naming Systems',
        'description': 'Shows Arabic naming conventions',
        'example_entity': 'محمد علي'
    },
    # Titles & Professional Context (2 types)
    'political_titles': {
        'name': 'Political Titles',
        'category': 'Titles & Professional Context',
        'description': 'Shows title + name matching',
        'example_entity': 'Xi Jinping'
    },
    'professional_titles': {
        'name': 'Professional Titles',
        'category': 'Titles & Professional Context',
        'description': 'Shows professional titles in Arabic',
        'example_entity': 'الدكتور أحمد'
    },
    # Organizations & Business (1-2 types)
    'company_name_variations': {
        'name': 'Company Name Variations',
        'category': 'Organizations & Business',
        'description': 'Shows multilingual company matching',
        'example_entity': 'トヨタ自動車'
    }
}

# Count challenge types from articles
article_challenge_types = {a.get('test_category', '') for a in tier5_articles}
print(f"📊 Selected {len(selected_challenge_types)} challenge types from {len(article_challenge_types)} total")
print(f"   Full dataset contains all {len(article_challenge_types)} challenge types covering multilingual, cultural, and business scenarios\n")


In [ ]:
# Helper function to get transliteration from entity aliases
def get_transliteration(entity_name, entities_list):
    """Get Latin transliteration from entity aliases"""
    for entity in entities_list:
        if entity['name'] == entity_name:
            aliases = entity.get('aliases', [])
            # Find first alias that's mostly Latin (ASCII) characters
            for alias in aliases:
                if alias and all(ord(c) < 128 for c in alias):
                    return alias
    return None

# Display showcase examples for each selected challenge type
showcase_results = []

# Group results by challenge type and create entity lookup
entity_to_challenge_type = {}
entity_lookup = {}  # name -> full entity dict
for entity in tier5_entities:
    entity_name = entity['name']
    challenge_type = entity.get('challenge_type', 'unknown')
    script = entity.get('script', 'unknown')
    entity_to_challenge_type[entity_name] = {
        'challenge_type': challenge_type,
        'script': script
    }
    entity_lookup[entity_name] = entity

# Find examples for each selected challenge type
for challenge_type_key, info in selected_challenge_types.items():
    print(f"\n{'='*60}")
    print(f"**{info['category']}: {info['name']}**")
    print(f"   Description: {info['description']}")
    
    # Find entity with this challenge type
    example_entity_name = info['example_entity']
    script = entity_to_challenge_type.get(example_entity_name, {}).get('script', 'unknown')
    print(f"   Script: {script}")
    
    # Get transliteration for the entity name
    transliteration = get_transliteration(example_entity_name, tier5_entities)
    
    # Find matching results for this entity
    entity_results = [r for r in all_results 
                     if r.get('watched_entity') == example_entity_name and r.get('is_match', False)]
    
    if entity_results:
        # Show first confirmed match
        example = entity_results[0]
        extracted = example.get('extracted_entity', '')
        watched = example.get('watched_entity', '')
        
        # Get transliterations for extracted and watched entities
        extracted_translit = get_transliteration(extracted, tier5_entities)
        watched_translit = get_transliteration(watched, tier5_entities)
        
        print(f"\n   ✅ Example Match:")
        
        # Show extracted entity with transliteration if available
        if extracted_translit and extracted != extracted_translit:
            print(f"      Extracted: '{extracted}' (transliteration: '{extracted_translit}')")
        else:
            print(f"      Extracted: '{extracted}'")
        
        # Show watched entity with transliteration if available
        if watched_translit and watched != watched_translit:
            print(f"      Matched:   '{watched}' (transliteration: '{watched_translit}')")
        else:
            print(f"      Matched:   '{watched}'")
        
        print(f"      Confidence: {example.get('confidence', 0.0):.2f}")
        print(f"      Match Type: {example.get('match_type', 'unknown')}")
        print(f"      Article ID: {example.get('article_id', 'unknown')}")
        
        # Add explanation for non-Latin characters
        if any(ord(c) > 127 for c in extracted + watched):
            print(f"\n   💡 What This Shows:")
            print(f"      This demonstrates {info['description'].lower()}")
            if script != 'Latin':
                print(f"      The {script} characters show how the system handles {script.lower()} script matching")
                if transliteration:
                    print(f"      The transliteration '{transliteration}' helps you understand what the characters represent")
        
        showcase_results.append({
            'challenge_type': challenge_type_key,
            'example': example
        })
    else:
        print(f"\n   ⚠️ No confirmed matches found for example entity '{example_entity_name}'")
        if transliteration:
            print(f"      (Entity name: '{example_entity_name}' = '{transliteration}' in transliteration)")

# Count challenge types from articles
article_challenge_types = {a.get('test_category', '') for a in tier5_articles}
print(f"\n{'='*60}")
print(f"\n📝 Note: The full Ultimate Challenge dataset contains {len(article_challenge_types)} challenge types.")
print(f"   We're showcasing {len(selected_challenge_types)} of the most educational and visually impressive examples.")
print(f"\n💡 Remember: You don't need to read the non-Latin characters to understand the concepts!")
print(f"   Focus on the challenge type (e.g., 'name order variations') and the transliterations provided.")


## 8. Results Analysis

This section breaks down results by key dimensions: script, challenge category, and match complexity. This analysis helps understand which types of entities are easiest/hardest to match and provides recommendations for production deployment.


In [ ]:
# Analyze performance by script
print("📊 Results Analysis: Performance by Script")
print("=" * 60)

script_stats = {}
for result in all_results:
    if not result.get('is_match', False):
        continue
    
    watched_entity = result.get('watched_entity', '')
    script = entity_to_challenge_type.get(watched_entity, {}).get('script', 'unknown')
    
    if script not in script_stats:
        script_stats[script] = {
            'total': 0,
            'confirmed': 0,
            'confidences': []
        }
    
    script_stats[script]['total'] += 1
    script_stats[script]['confirmed'] += 1 if result.get('is_match', False) else 0
    script_stats[script]['confidences'].append(result.get('confidence', 0.0))

print("\n**Script Performance:**")
for script in sorted(script_stats.keys()):
    stats = script_stats[script]
    avg_confidence = sum(stats['confidences']) / len(stats['confidences']) if stats['confidences'] else 0.0
    print(f"\n   {script}:")
    print(f"      Total matches: {stats['total']}")
    print(f"      Average confidence: {avg_confidence:.2f}")


In [ ]:
# Analyze performance by challenge category
print("\n📊 Results Analysis: Performance by Challenge Category")
print("=" * 60)

# Map challenge types to categories
challenge_category_map = {
    'multilingual': ['japanese_transliteration', 'chinese_transliteration', 'cross_script_transliteration', 
                     'advanced_transliteration', 'korean_transliteration', 'multiple_writing_systems'],
    'cultural': ['name_order_variations', 'patronymic_systems', 'honorific_systems'],
    'titles': ['political_titles', 'professional_titles', 'religious_titles', 'business_titles', 
               'academic_titles', 'military_political_titles'],
    'organizations': ['company_name_variations', 'company_abbreviations', 'company_variations',
                      'complex_business_hierarchy', 'parent_subsidiary_relationships',
                      'historical_name_changes', 'business_diversification']
}

category_stats = {}
for result in all_results:
    if not result.get('is_match', False):
        continue
    
    watched_entity = result.get('watched_entity', '')
    challenge_type = entity_to_challenge_type.get(watched_entity, {}).get('challenge_type', 'unknown')
    
    # Determine category
    category = 'other'
    for cat, types in challenge_category_map.items():
        if challenge_type in types:
            category = cat
            break
    
    if category not in category_stats:
        category_stats[category] = {
            'total': 0,
            'confidences': []
        }
    
    category_stats[category]['total'] += 1
    category_stats[category]['confidences'].append(result.get('confidence', 0.0))

print("\n**Challenge Category Performance:**")
for category in sorted(category_stats.keys()):
    stats = category_stats[category]
    avg_confidence = sum(stats['confidences']) / len(stats['confidences']) if stats['confidences'] else 0.0
    print(f"\n   {category.title()}:")
    print(f"      Total matches: {stats['total']}")
    print(f"      Average confidence: {avg_confidence:.2f}")


In [ ]:
# Analyze performance by match type
print("\n📊 Results Analysis: Performance by Match Type")
print("=" * 60)

match_type_stats = {}
for result in all_results:
    match_type = result.get('match_type', 'unknown')
    
    if match_type not in match_type_stats:
        match_type_stats[match_type] = {
            'total': 0,
            'confirmed': 0,
            'confidences': []
        }
    
    match_type_stats[match_type]['total'] += 1
    if result.get('is_match', False):
        match_type_stats[match_type]['confirmed'] += 1
        match_type_stats[match_type]['confidences'].append(result.get('confidence', 0.0))

print("\n**Match Type Distribution:**")
for match_type in sorted(match_type_stats.keys()):
    stats = match_type_stats[match_type]
    avg_confidence = sum(stats['confidences']) / len(stats['confidences']) if stats['confidences'] else 0.0
    confirmation_rate = stats['confirmed'] / stats['total'] if stats['total'] > 0 else 0.0
    
    print(f"\n   {match_type}:")
    print(f"      Total: {stats['total']}")
    print(f"      Confirmed: {stats['confirmed']} ({confirmation_rate*100:.1f}%)")
    print(f"      Average confidence: {avg_confidence:.2f}")

print("\n**Insights and Recommendations:**")
print("   • Exact matches typically have highest confidence and confirmation rates")
print("   • Transliteration matches require more sophisticated handling but show good results")
print("   • Cultural naming variations benefit from explicit context in watch list")
print("   • Production deployment should account for script-specific challenges")
print("   • Consider different confidence thresholds per challenge category")


## 9. Conclusion & Takeaways

This capstone notebook has demonstrated the entity resolution system's ultimate capabilities on the Ultimate Challenge dataset. Key learnings and next steps are summarized below.


In [ ]:
print("🎉 Ultimate Challenge Capstone Complete!")
print("=" * 60)

print("\n**Key Learnings:**")
print("   ✅ Direct matching approach successfully bypasses event loop conflicts")
print("   ✅ MinimalFunctionCallingJudge handles multilingual matching effectively")
print("   ✅ System demonstrates strong performance across 6+ scripts")
print("   ✅ Quality metrics (precision, recall, F1) validate system capabilities")
print("   ✅ Challenge type showcase highlights impressive multilingual capabilities")

print("\n**System Capabilities Demonstrated:**")
print("   • Multilingual entity matching (Japanese, Arabic, Hebrew, Chinese, Cyrillic, Korean, Latin)")
print("   • Cultural naming convention handling (name order, patronymic systems)")
print("   • Professional and political title matching")
print("   • Business entity resolution (company variations, historical name changes)")
print("   • Cross-script transliteration")

print("\n**Next Steps for Production Use:**")
print("   1. Consider script-specific confidence thresholds")
print("   2. Implement challenge category-specific handling")
print("   3. Monitor match type distribution for quality assurance")
print("   4. Use explicit_context in watch list for complex entities")
print("   5. Regular evaluation against golden standards")

print(f"\n✅ F1 Score: {f1:.3f} ({f1*100:.1f}%)")
if f1 >= 0.85:
    print("   🎯 Performance exceeds expected threshold - ready for production!")
else:
    print("   ⚠️ Consider further optimization before production deployment")

print("\n" + "=" * 60)
